In [55]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data_loader import load_data

from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder,
)

In [56]:
df = load_data("../data/raw/dataset.csv") 

In [57]:
# antes de fazer a limpeza, vamos criar uma cópia do dataframe original para não perder os dados originais
df_limpo = df.copy()

In [58]:
# Identificando as colunas de texto
colunas_texto = df_limpo.select_dtypes(include="str").columns
colunas_texto

# Convertendo para string e removendo espaços extras
for col in colunas_texto:
    df_limpo[col] = df_limpo[col].astype(str).str.strip()

print("Espaços removidos das colunas de texto.")

Espaços removidos das colunas de texto.


In [59]:
# Colunas categóricas com valores ausentes
colunas_nao_informadas = [
    "favorite_game_genre",
    "favorite_music_genre"
]

for col in colunas_nao_informadas:
    # Restaurando os valores "nan" como valores ausentes do Pandas
    df_limpo[col] = df_limpo[col].replace("nan", np.nan)

    # Tratando a ausência como uma categoria explícita
    df_limpo[col] = df_limpo[col].fillna("Nao_Informado")

print("Valores ausentes tratados como 'Nao_Informado'.")

Valores ausentes tratados como 'Nao_Informado'.


In [60]:
def verificar_categorias(df):
    colunas_texto = df.select_dtypes(include="object").columns

    for col in colunas_texto:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))

In [61]:
verificar_categorias(df_limpo)


age_group
age_group
Adult (26-40)          1858
Young Adult (18-25)    1494
Middle Age (41-60)     1137
Teen (13-17)            893
Senior (60+)            618
Name: count, dtype: int64

gender
gender
Male          2833
Female        2809
Non-binary     358
Name: count, dtype: int64

personality_type
personality_type
Extrovert    2256
Introvert    1987
Ambivert     1757
Name: count, dtype: int64

hobbies
hobbies
Writing                               157
Cooking                               154
Watching TV                           153
Gardening                             153
Sports                                147
                                     ... 
Music, Photography, Dancing             1
Photography, Painting/Art, Cooking      1
Gardening, Writing, Fitness             1
Hiking, Gardening, Cooking              1
Watching TV, Painting/Art, Fitness      1
Name: count, Length: 1529, dtype: int64

favorite_game_genre
favorite_game_genre
Action             952
RPG              

C:\Users\lucas\AppData\Local\Temp\ipykernel_13568\2944076800.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


Tratamento de outliers

Os possíveis outliers identificados na etapa de EDA foram avaliados
considerando o contexto das variáveis.

mood_score:
Os valores estão dentro da escala esperada de 1 a 10.
Portanto, os valores extremos foram mantidos.

social_media_hours_per_day:
Os valores identificados como possíveis outliers pelo IQR
representam horas de uso que, embora elevadas, são plausíveis.
Portanto, também foram mantidos.

Não foram removidas observações do dataset.

In [62]:
df_limpo.shape

(6000, 35)

In [63]:
colunas_categoricas = df_limpo.select_dtypes(
    include="str"
).columns.tolist()

colunas_categoricas.remove("hobbies")
colunas_categoricas.remove("recommended_movie_genre")

print(colunas_categoricas)

['age_group', 'gender', 'personality_type', 'favorite_game_genre', 'favorite_music_genre', 'preferred_language', 'relationship_status', 'favorite_season', 'sleep_pattern', 'education_level', 'occupation', 'income_bracket', 'preferred_streaming_platform', 'preferred_watch_time', 'viewing_companion', 'snack_preference', 'preferred_device', 'preferred_movie_length', 'favorite_movie_era', 'exercise_frequency', 'pet_ownership', 'travel_frequency', 'book_reading_frequency']


Variáveis numéricas: mood_score, stress_level, fear_tolerance, risk_tolerance, humor_preference, openness_score, romance_interest, social_media_hours_per_day, streaming_subscription_count

Variáveis nominal categóricas: gender, personality_type, hobbies, favorite_game_genre, favorite_music_genre, preferred_language, relationship_status, favorite_season, sleep_pattern, occupation, preferred_streaming_platform, preferred_watch_time, viewing_companion, snack_preference, preferred_device, favorite_movie_era, pet_ownership, recommended_movie_genre, A few books a year

Variáveis ordinal categóricas: age_group, education_level, income_bracket, exercise_frequency, preferred_movie_length, travel_frequency

In [64]:
ordem_age_group = [
    "Teen (13-17)",
    "Young Adult (18-25)",
    "Adult (26-40)",
    "Middle Age (41-60)",
    "Senior (60+)"
]

encoder_ordinal = OrdinalEncoder(categories=[ordem_age_group])

df_limpo["age_group_encoded"] = encoder_ordinal.fit_transform(
    df_limpo[["age_group"]]
)

df_limpo[["age_group", "age_group_encoded"]].drop_duplicates().sort_values(
    "age_group_encoded"
)

,age_group,age_group_encoded
0,Teen (13-17),0.0
1,Young Adult (18-25),1.0
13,Adult (26-40),2.0
24,Middle Age (41-60),3.0
7,Senior (60+),4.0


In [65]:
ordem_education = [
    "High School",
    "Vocational/Diploma",
    "Bachelor's Degree",
    "Master's Degree",
    "PhD"
]

encoder_education = OrdinalEncoder(categories=[ordem_education])

df_limpo["education_level_encoded"] = encoder_education.fit_transform(
    df_limpo[["education_level"]]
)

df_limpo[["education_level", "education_level_encoded"]].drop_duplicates().sort_values(
    "education_level_encoded"
)

,education_level,education_level_encoded
1,High School,0.0
3,Vocational/Diploma,1.0
2,Bachelor's Degree,2.0
0,Master's Degree,3.0
7,PhD,4.0


In [66]:
ordem_income = [
    "Low (<$25k)",
    "Lower-Middle ($25k-50k)",
    "Middle ($50k-90k)",
    "Upper-Middle ($90k-150k)",
    "High (>$150k)"
]

encoder_ordinal = OrdinalEncoder(categories=[ordem_income])

df_limpo["income_bracket_encoded"] = encoder_ordinal.fit_transform(
    df_limpo[["income_bracket"]]
)

df_limpo[["income_bracket", "income_bracket_encoded"]].drop_duplicates().sort_values(
    "income_bracket_encoded"
)

,income_bracket,income_bracket_encoded
3,Low (<$25k),0.0
1,Lower-Middle ($25k-50k),1.0
0,Middle ($50k-90k),2.0
5,Upper-Middle ($90k-150k),3.0
4,High (>$150k),4.0


In [33]:
ordem_exercise = [
    "Never",
    "Rarely",
    "1-2 times/week",
    "3-5 times/week",
    "Daily"
]

encoder_exercise = OrdinalEncoder(categories=[ordem_exercise])

df_limpo["exercise_frequency_encoded"] = encoder_exercise.fit_transform(
    df_limpo[["exercise_frequency"]]
)

df_limpo[["exercise_frequency", "exercise_frequency_encoded"]].drop_duplicates().sort_values(
    "exercise_frequency_encoded"
)

,exercise_frequency,exercise_frequency_encoded
8,Never,0.0
2,Rarely,1.0
5,1-2 times/week,2.0
0,3-5 times/week,3.0
6,Daily,4.0


In [67]:
ordem_movie_length = [
    "Short (<90 min)",
    "Standard (90-120 min)",
    "Long (120-160 min)",
    "Epic (160+ min)"
]

encoder_movie_length = OrdinalEncoder(categories=[ordem_movie_length])

df_limpo["preferred_movie_length_encoded"] = encoder_movie_length.fit_transform(
    df_limpo[["preferred_movie_length"]]
)

df_limpo[["preferred_movie_length", "preferred_movie_length_encoded"]].drop_duplicates().sort_values(
    "preferred_movie_length_encoded"
)

,preferred_movie_length,preferred_movie_length_encoded
3,Short (<90 min),0.0
0,Standard (90-120 min),1.0
11,Long (120-160 min),2.0
13,Epic (160+ min),3.0


In [70]:
ordem_travel = [
    "Never",
    "Once a year",
    "2-3 times/year",
    "Monthly",
    "Frequently (weekly/biweekly)"
]

encoder_travel = OrdinalEncoder(categories=[ordem_travel])

df_limpo["travel_frequency_encoded"] = encoder_travel.fit_transform(
    df_limpo[["travel_frequency"]]
)

df_limpo[["travel_frequency", "travel_frequency_encoded"]].drop_duplicates().sort_values(
    "travel_frequency_encoded"
)

,travel_frequency,travel_frequency_encoded
22,Never,0.0
3,Once a year,1.0
0,2-3 times/year,2.0
2,Monthly,3.0
1,Frequently (weekly/biweekly),4.0


In [71]:
colunas_nominais = [
    "gender",
    "personality_type",
    "favorite_game_genre",
    "favorite_music_genre",
    "preferred_language",
    "relationship_status",
    "favorite_season",
    "sleep_pattern",
    "occupation",
    "preferred_streaming_platform",
    "preferred_watch_time",
    "viewing_companion",
    "snack_preference",
    "preferred_device",
    "favorite_movie_era",
    "pet_ownership",
    "book_reading_frequency"
]

In [72]:
df_one_hot = pd.get_dummies(
    df_limpo[colunas_nominais],
    prefix=colunas_nominais,
    dtype=int
)

df_limpo = pd.concat(
    [df_limpo, df_one_hot],
    axis=1
)

In [73]:
print("Colunas antes do One-Hot:", len(df_limpo.columns) - len(df_one_hot.columns))
print("Novas colunas criadas:", len(df_one_hot.columns))
print("Total de colunas:", len(df_limpo.columns))

Colunas antes do One-Hot: 40
Novas colunas criadas: 94
Total de colunas: 134


In [74]:
df_limpo.head()

,person_id,age_group,gender,personality_type,mood_score,stress_level,hobbies,favorite_game_genre,favorite_music_genre,fear_tolerance,...,pet_ownership_Has Cat,pet_ownership_Has Dog,pet_ownership_Has Other Pet,pet_ownership_Multiple Pets,pet_ownership_No Pets,book_reading_frequency_1-2 books/month,book_reading_frequency_A few books a year,book_reading_frequency_Avid reader (3+ books/month),book_reading_frequency_Never,book_reading_frequency_Weekly reader
0,1,Teen (13-17),Male,Ambivert,6.0,4.5,Music,Simulation,Indie,1.9,...,0,0,0,0,1,1,0,0,0,0
1,2,Young Adult (18-25),Female,Extrovert,7.5,5.8,"Reading, Traveling",Horror/Survival,Hip-Hop,2.7,...,0,0,0,0,1,0,0,0,1,0
2,3,Young Adult (18-25),Male,Ambivert,7.2,6.9,"Fitness, Painting/Art, Gaming",Puzzle,Rock,7.1,...,0,1,0,0,0,0,1,0,0,0
3,4,Teen (13-17),Male,Introvert,5.2,2.5,"Painting/Art, Writing, Sports",Puzzle,Nao_Informado,3.6,...,1,0,0,0,0,0,0,0,1,0
4,5,Teen (13-17),Male,Introvert,6.3,8.8,Cooking,Puzzle,Pop,7.5,...,0,0,0,0,1,0,0,1,0,0


In [77]:
label_encoder = LabelEncoder()

df_limpo["recommended_movie_genre_encoded"] = label_encoder.fit_transform(
    df_limpo["recommended_movie_genre"]
)

df_limpo[["recommended_movie_genre", "recommended_movie_genre_encoded"]].drop_duplicates().sort_values(
    "recommended_movie_genre_encoded"
)

,recommended_movie_genre,recommended_movie_genre_encoded
3,Action,0
56,Animation,1
2,Comedy,2
15,Documentary,3
4,Drama,4
46,Horror,5
1,Romance,6
7,Sci-Fi,7
0,Thriller,8


In [78]:
df_limpo["hobbies"].value_counts().head(30)

hobbies
Writing                  157
Cooking                  154
Watching TV              153
Gardening                153
Sports                   147
Gaming                   143
Traveling                142
Dancing                  138
Photography              137
Hiking                   134
Painting/Art             133
Reading                  133
Fitness                  131
Music                    130
Reading, Fitness          26
Traveling, Fitness        22
Dancing, Gardening        18
Dancing, Gaming           17
Reading, Gaming           17
Watching TV, Dancing      17
Watching TV, Fitness      17
Cooking, Reading          17
Hiking, Cooking           16
Traveling, Cooking        16
Watching TV, Hiking       16
Dancing, Photography      16
Traveling, Gaming         16
Dancing, Hiking           16
Painting/Art, Cooking     16
Music, Hiking             16
Name: count, dtype: int64

In [79]:
hobbies_unicos = (
    df_limpo["hobbies"]
    .dropna()
    .str.split(", ")
    .explode()
    .unique()
)

len(hobbies_unicos), sorted(hobbies_unicos)

(14,
 ['Cooking',
  'Dancing',
  'Fitness',
  'Gaming',
  'Gardening',
  'Hiking',
  'Music',
  'Painting/Art',
  'Photography',
  'Reading',
  'Sports',
  'Traveling',
  'Watching TV',
  'Writing'])

In [80]:
hobbies_unicos = (
    df_limpo["hobbies"]
    .dropna()
    .str.split(", ")
    .explode()
    .unique()
)

for hobby in hobbies_unicos:
    df_limpo[f"hobby_{hobby}"] = (
        df_limpo["hobbies"]
        .fillna("")
        .str.contains(hobby, regex=False)
        .astype(int)
    )

In [81]:
[c for c in df_limpo.columns if c.startswith("hobby_")]

['hobby_Music',
 'hobby_Reading',
 'hobby_Traveling',
 'hobby_Fitness',
 'hobby_Painting/Art',
 'hobby_Gaming',
 'hobby_Writing',
 'hobby_Sports',
 'hobby_Cooking',
 'hobby_Dancing',
 'hobby_Hiking',
 'hobby_Photography',
 'hobby_Gardening',
 'hobby_Watching TV']

In [82]:
df_limpo.head()

,person_id,age_group,gender,personality_type,mood_score,stress_level,hobbies,favorite_game_genre,favorite_music_genre,fear_tolerance,...,hobby_Painting/Art,hobby_Gaming,hobby_Writing,hobby_Sports,hobby_Cooking,hobby_Dancing,hobby_Hiking,hobby_Photography,hobby_Gardening,hobby_Watching TV
0,1,Teen (13-17),Male,Ambivert,6.0,4.5,Music,Simulation,Indie,1.9,...,0,0,0,0,0,0,0,0,0,0
1,2,Young Adult (18-25),Female,Extrovert,7.5,5.8,"Reading, Traveling",Horror/Survival,Hip-Hop,2.7,...,0,0,0,0,0,0,0,0,0,0
2,3,Young Adult (18-25),Male,Ambivert,7.2,6.9,"Fitness, Painting/Art, Gaming",Puzzle,Rock,7.1,...,1,1,0,0,0,0,0,0,0,0
3,4,Teen (13-17),Male,Introvert,5.2,2.5,"Painting/Art, Writing, Sports",Puzzle,Nao_Informado,3.6,...,1,0,1,1,0,0,0,0,0,0
4,5,Teen (13-17),Male,Introvert,6.3,8.8,Cooking,Puzzle,Pop,7.5,...,0,0,0,0,1,0,0,0,0,0


In [83]:
df_limpo.to_csv(
    "../data/processed/dataset_clean.csv",
    index=False
)